# Get the campaign list

*Data Analysis in Python · Topic 12.1*

Your media team wants to know how many campaigns each channel ran. Ask the Simmer API for the campaign list, and count the campaigns per channel.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Add the course API key to Colab Secrets, if you haven't yet: open the key icon in the left bar, choose **Add new secret**, name it `SIMMER_API_KEY`, paste the key from the topic's Topic Materials, and switch on **Notebook access**.
2. Run the parameters cell.
3. In the first cell marked ✏️, send a request to the `campaigns` endpoint without the key, and create `no_key_status`: the status code of its answer.
4. In the second cell marked ✏️, send the same request with the key in the `X-API-Key` header, and call `raise_for_status()`. Then create the `campaigns` table from the answer's `campaigns` list, and `campaigns_per_channel`: the number of campaigns in each `channel`.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `no_key_status` is `401`. The `campaigns` table has 104 campaigns. `campaigns_per_channel` has 5 channels, from 36 Paid Social campaigns to 8 Paid Video campaigns.

## Parameters

In [ ]:
import pandas as pd
import requests
from google.colab import userdata

BASE_URL = "https://api.teamsimmer.com/v1/"
API_KEY = userdata.get("SIMMER_API_KEY")  # Colab Secrets: the key icon in the left bar

## Without the key

In [ ]:
# ✏️ Create no_key_status


## With the key

In [ ]:
# ✏️ Create campaigns and campaigns_per_channel


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    if not found.get("API_KEY"):
        results.append((False, "API_KEY is empty. Add the key to Colab Secrets as SIMMER_API_KEY, switch on Notebook access, and run the parameters cell again."))

    status = found.get("no_key_status")
    if status is None:
        results.append((False, "There's no no_key_status yet: requests.get(BASE_URL + \"campaigns\").status_code."))
    elif status == 200:
        results.append((False, "no_key_status is 200: that request sent the key. Send the first request without headers."))
    elif status != 401:
        results.append((False, f"no_key_status should be 401, the status code of a request without the key. It is {status!r}."))
    else:
        results.append((True, "Without the key, the API answers 401."))

    campaigns = found.get("campaigns")
    if not isinstance(campaigns, pd.DataFrame):
        results.append((False, "campaigns should be a DataFrame: pd.DataFrame(body[\"campaigns\"])."))
    elif "channel" not in campaigns.columns:
        results.append((False, "The campaigns table should have one row per campaign, with a channel column. Make it from the answer's campaigns list, not from the whole answer."))
    elif len(campaigns) != 104:
        results.append((False, f"The campaigns table should have the 104 campaigns. It has {len(campaigns)}."))
    else:
        results.append((True, "104 campaigns, with their channel, platform and dates."))

    per_channel = found.get("campaigns_per_channel")
    expected = {"Paid Social": 36, "Paid Search": 28, "Email": 20, "Display": 12, "Paid Video": 8}
    if not isinstance(per_channel, pd.Series):
        results.append((False, "campaigns_per_channel should be a Series: campaigns[\"channel\"].value_counts()."))
    elif {str(name): int(n) for name, n in per_channel.items()} != expected:
        results.append((False, f"campaigns_per_channel should count the campaigns in each channel: {expected}. It is {dict(per_channel)}."))
    else:
        results.append((True, "36 Paid Social campaigns, 28 Paid Search, 20 Email, 12 Display and 8 Paid Video."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
no_key_status = requests.get(BASE_URL + "campaigns").status_code
no_key_status
```

```python
response = requests.get(BASE_URL + "campaigns", headers={"X-API-Key": API_KEY})
response.raise_for_status()
campaigns = pd.DataFrame(response.json()["campaigns"])
campaigns_per_channel = campaigns["channel"].value_counts()
campaigns_per_channel
```

</details>